<a href="https://colab.research.google.com/github/GodishalaAshwith/DeepLearning/blob/main/DLAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading Dataset

In [4]:
import kagglehub
path = kagglehub.dataset_download("leoscode/wound-segmentation-images")

Using Colab cache for faster access to the 'wound-segmentation-images' dataset.


In [5]:
import os

print("Dataset Path:", path)
print(os.listdir(path))

Dataset Path: /kaggle/input/wound-segmentation-images
['data_wound_seg']


In [6]:
for root, dirs, files in os.walk(path):
    print("Root:", root)
    print("Dirs:", dirs)
    print("Files:", len(files))
    break

Root: /kaggle/input/wound-segmentation-images
Dirs: ['data_wound_seg']
Files: 0


In [8]:
base_path = path + "/data_wound_seg"

train_img_path = base_path + "/train_images"
train_mask_path = base_path + "/train_masks"

test_img_path = base_path + "/test_images"
test_mask_path = base_path + "/test_masks"

In [9]:
import os
import cv2
import numpy as np
import shutil

dataset_path = "/content/dataset"
train_dir = dataset_path + "/train"
val_dir = dataset_path + "/val"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

from sklearn.model_selection import train_test_split

images = os.listdir(train_img_path)

train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

def process_and_copy(img_list, target_dir):
    for img_name in img_list:
        img_path = os.path.join(train_img_path, img_name)
        mask_path = os.path.join(train_mask_path, img_name)

        mask = cv2.imread(mask_path, 0)  # grayscale

        # Check if wound exists
        if np.sum(mask) > 0:
            label = "wound"
        else:
            label = "normal"

        os.makedirs(os.path.join(target_dir, label), exist_ok=True)

        shutil.copy(img_path, os.path.join(target_dir, label, img_name))

# Create dataset
process_and_copy(train_imgs, train_dir)
process_and_copy(val_imgs, val_dir)

In [18]:
for cls in os.listdir(train_dir):
    print(cls, ":", len(os.listdir(os.path.join(train_dir, cls))))

normal : 1000
wound : 1757


In [11]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

print("Classes:", train_data.classes)

Classes: ['normal', 'wound']


In [12]:
import random
import shutil

normal_path = train_dir + "/normal"
normal_images = os.listdir(normal_path)

target_count = 1000  # match closer to wound count

for i in range(target_count - len(normal_images)):
    img = random.choice(normal_images)

    src = os.path.join(normal_path, img)
    dst = os.path.join(normal_path, f"aug_{i}_{img}")

    shutil.copy(src, dst)

In [13]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [15]:
import torch

# Define device before use
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_counts = [len(os.listdir(train_dir + "/normal")),
                len(os.listdir(train_dir + "/wound"))]

weights = torch.tensor([
    1.0/class_counts[0],
    1.0/class_counts[1]
])

weights = weights / weights.sum()

criterion = torch.nn.CrossEntropyLoss(weight=weights.to(device))

In [17]:
from torch.utils.data import WeightedRandomSampler

targets = train_data.targets
# Convert targets to a torch.Tensor for element-wise operations
targets_tensor = torch.tensor(targets)

class_sample_count = torch.tensor([
    (targets_tensor == 0).sum(),
    (targets_tensor == 1).sum()
], dtype=torch.float)

weights = 1. / class_sample_count
sample_weights = weights[targets_tensor] # Use targets_tensor here as well

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_data, batch_size=32, sampler=sampler)

In [19]:
selected_wounds = random.sample(os.listdir(train_dir + "/wound"), 1000)

In [20]:
for cls in os.listdir(train_dir):
    print(cls, ":", len(os.listdir(os.path.join(train_dir, cls))))

normal : 1000
wound : 1757


In [21]:
import os
import random

wound_path = train_dir + "/wound"

all_wounds = os.listdir(wound_path)
selected_wounds = random.sample(all_wounds, 1000)

# Remove extra images
for img in all_wounds:
    if img not in selected_wounds:
        os.remove(os.path.join(wound_path, img))

In [22]:
for cls in os.listdir(train_dir):
    print(cls, ":", len(os.listdir(os.path.join(train_dir, cls))))

normal : 1000
wound : 1000


In [23]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

print(train_data.classes)

['normal', 'wound']


# Training

### ResNet

In [25]:
import torchvision.models as models
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 67.7MB/s]


In [26]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

In [27]:
for epoch in range(10):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 4.6028
Epoch 2, Loss: 0.3679
Epoch 3, Loss: 1.1467
Epoch 4, Loss: 0.6574
Epoch 5, Loss: 0.6355
Epoch 6, Loss: 0.0942
Epoch 7, Loss: 0.0105
Epoch 8, Loss: 0.0078
Epoch 9, Loss: 0.0055
Epoch 10, Loss: 0.0067


In [28]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:", 100 * correct / total)

Validation Accuracy: 99.5475113122172


In [29]:
from sklearn.metrics import confusion_matrix
import numpy as np

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[  0   2]
 [  0 440]]
